# Import & Config

In [1]:
import time
import random
import os
import pickle
import itertools
import gc

# 데이터 분석 라이브러리
import numpy as np
import pandas as pd
import gzip
import zipfile
import math
from tqdm import tqdm
from tqdm import trange
tqdm.pandas()  # <- 이걸 실행해야 progress_apply가 활성화됨

# 케라스 라이브러리
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, Flatten, Dot, Multiply, Concatenate, Dropout, Dense, Add, Activation,
    MultiHeadAttention, Attention, Lambda, GlobalMaxPooling1D, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# 데이터 분할, 인코더, 성능 평가 라이브러리
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import ParameterGrid

print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')
print(f'tensorflow version: {tf.__version__}')       # 2.13.0 등

pandas version: 1.5.3
numpy version: 1.23.5
tensorflow version: 2.12.0


In [ ]:
DATA_SAVE_PATH = 'C:/Users/Bigdata64/Desktop/MK/dataset'
CATEGORY = 'yelp'
DATASET_NM = 'yelp'

# 변수 설정
dict_col_nm = {
   'amazon': {
        'USER_ID': 'userID',
        'ITEM_ID': 'itemID',
        'RATING_COL': 'rating',
        'REVIEW_COL': 'reviewtext'
      },
   'yelp': {
        'USER_ID': 'user_id',
        'ITEM_ID': 'rest_id',
        'RATING_COL': 'review_stars',
        'REVIEW_COL': 'review_text'
      }
}

# 평균 패딩 시퀀스 길이 할당
dict_mean_seq_len = {
    'book':118,
    'movie':44,
    'yelp':113,
    'office': 51
}

USER_COL = dict_col_nm[DATASET_NM]['USER_ID']
ITEM_COL = dict_col_nm[DATASET_NM]['ITEM_ID']
RATING_COL = dict_col_nm[DATASET_NM]['RATING_COL']
REVIEW_COL = dict_col_nm[DATASET_NM]['REVIEW_COL']

mean_seq_length = dict_mean_seq_len[CATEGORY]

print(f"{CATEGORY} average seq length: {mean_seq_length}")

train_path = f'{DATA_SAVE_PATH}/{CATEGORY}_train_dict.pkl'
test_path = f'{DATA_SAVE_PATH}/{CATEGORY}_test_dict.pkl'

yelp average seq length: 113


In [ ]:
# hyperparameter 설정
default_params = {
    'learning_rate': 1e-4,
    'batch_size': 128,
    'dropout_rate': 0.2,
    'mlp_hidden_dim': 128,
    'mlp_depth': 2,
    'embedding_dim': 300,
    'input_dim': mean_seq_length,
    'fusion_version': 'gmu'
}

In [3]:
def merge_textrank_bart(path, which_user_item, id_col):
    # load/rename textrank
    df_textrank = pd.read_pickle(path + which_user_item+'_textrank.pkl')

    df_textrank = df_textrank.rename(columns = {
        id_col:which_user_item,
        f'{which_user_item}_summary_padded_sequences':f'{which_user_item}_embedding_textrank'})

    # load bart
    df_bart = pd.read_pickle(path + which_user_item+'_bart.pkl')

    ## prep id column
    if which_user_item+'_id' in df_bart.columns:
        df_bart = df_bart.rename(columns = {which_user_item+'_id':which_user_item})
    if id_col in df_bart.columns:
        df_bart = df_bart.rename(columns = {id_col:'item'})

    ## rename bart column
    df_bart = df_bart.rename(columns = {'bart_embedding':f'{which_user_item}_embedding_bart'})

    # merge
    df_merge = pd.merge(
        df_textrank[[which_user_item, f'{which_user_item}_embedding_textrank']],
        df_bart[[which_user_item, f'{which_user_item}_embedding_bart']],
        on = which_user_item)

    return df_merge

# Merge
- Textrank, BART 데이터 결합

In [ ]:
df_user_final = merge_textrank_bart(
    path = f"{DATA_SAVE_PATH}/{CATEGORY}_", which_user_item = 'user', id_col = USER_ID)

print(df_user_final.shape)
df_user_final.head(5)

(32927, 3)


,user,user_embedding_textrank,user_embedding_bart
0,--4AjktZiHowEIBCMd4CZA,"[1904, 302, 214, 1, 160, 2, 1394, 202, 483, 30...","[0.7941786, 1.2260883, -0.6717911, -5.146889, ..."
1,--F7iEaFFPO1UYemj-dUNw,"[259, 17, 51, 1, 6590, 266, 2, 10, 26, 3, 1667...","[-0.10013671, -0.76346076, -1.4995432, -0.5768..."
2,--_r6E98SNIrGU7weyNxbw,"[1, 149, 44, 1267, 10470, 4, 128, 4, 408, 34, ...","[-0.6863516, -0.46089366, 0.024552483, -1.3864..."
3,--ccVMj2PN6Z9qtdOdlung,"[1, 14, 4, 558, 106, 9, 8, 4639, 3805, 11, 3, ...","[-0.94170034, -1.1661474, 1.3066427, -1.689525..."
4,--dVKamoZnV2vYwKtWMVVA,"[3154, 26623, 33, 3, 190, 7, 1, 85, 1039, 182,...","[-1.2513263, -0.21510649, -2.2218053, -2.42989..."


In [ ]:
df_item_final = merge_textrank_bart(
    path = f"{DATA_SAVE_PATH}/{CATEGORY}_", which_user_item = 'item', id_col = ITEM_ID)

print(df_item_final.shape)
df_item_final.head(5)

(7144, 3)


,item,item_embedding_textrank,item_embedding_bart
0,--epgcb7xHGuJ-4PUeSLAw,"[138, 32, 93, 32, 132, 134, 929, 1, 2410, 4, 1...","[-1.2977682, -1.0983266, -0.07653178, -3.85488..."
1,-0FX23yAacC4bbLaGPvyxw,"[45, 379, 20, 1, 461, 1, 14, 533, 416, 12298, ...","[-0.9406209, -1.1824609, -0.46529746, -1.85860..."
2,-0TffRSXXIlBYVbb5AwfTg,"[29, 64, 13, 3, 626, 2243, 5, 599, 410, 2, 146...","[-0.72921675, -0.62988174, 0.40293264, -2.6648..."
3,-1B9pP_CrRBJYPICE5WbRA,"[8, 36, 109, 2, 16, 55, 6, 581, 314, 292, 3, 9...","[1.0505005, 1.1355219, -1.3780993, -0.11218664..."
4,-3725FZiIIYdwQtM4MKEIA,"[7, 14914, 68, 1933, 12, 37, 739, 1, 7096, 230...","[-0.23185682, -0.9555145, 0.12665835, -0.22444..."


In [ ]:
# save user/item textrank(included padded_seq data)
df_user_final.to_pickle(f"{DATA_SAVE_PATH}/{CATEGORY}_user_textrank_padded.pkl")
df_item_final.to_pickle(f"{DATA_SAVE_PATH}/{CATEGORY}_item_textrank_padded.pkl")

In [ ]:
# load rating, userID, itemID
df_all = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_data.pkl')
df_all = df_all[[USER_ID, ITEM_ID, RATING_COL]].copy()
df_all = df_all.rename(columns = {USER_ID:'user', ITEM_ID:'item', RATING_COL:'rating'})
df_all.head(5)

,user,item,rating
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0
2,mh_-eMZ6K5RLWhZyISBhwA,mOk3D7VczrDapNuUgLxUQw,4.0
3,mh_-eMZ6K5RLWhZyISBhwA,L4kfcADLCU4T33i7Z0CkuA,2.0
4,mh_-eMZ6K5RLWhZyISBhwA,tVfJPW14AeuAHDJeleWWdQ,3.0
5,mh_-eMZ6K5RLWhZyISBhwA,gpYBhnTk4KzvvH83TsZiQg,5.0


In [ ]:
# merge user/item data
df_final_input = pd.merge(df_all, df_user_final, on = 'user')
df_final_input = pd.merge(df_final_input, df_item_final, on = 'item')

print(df_final_input.shape)
df_final_input.head(5)

(472877, 7)


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[304, 42, 8, 3, 29, 370, 231, 2, 1, 536, 30, 9...","[-1.598509, -0.997596, 0.5327545, -1.525021, 1...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
1,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,"[46, 898, 383, 71, 44, 14, 49, 8, 32, 19, 32, ...","[-1.4743725, -1.3923125, -0.2920361, -2.608637...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
2,f7xa0p_1V9lx53iIGN5Sug,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[21, 52, 3, 190, 7, 14, 1, 121, 188, 919, 8, 3...","[-0.21176815, -0.2568603, -1.0278563, -2.40477...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
3,dCooFVCk8M1nVaQqcfTL3Q,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[39, 1, 470, 6, 642, 2, 2531, 2, 4, 429, 33, 4...","[0.82523656, 0.91178995, 0.60730594, -4.156382...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
4,gy7Ss1uTpCjbbGsghTvNsw,XQfwVwDr-v0ZS3_CbbE5Xw,2.0,"[9, 3, 208, 119, 16, 22, 26, 19, 382, 17, 64, ...","[-2.3661036, -0.6385349, 0.5271855, -3.6942627...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."


In [ ]:
## 확인용: 사용자별 아이템 수(max, min, mean) -> 5-core
df_num_items_per_user = df_final_input.groupby('user')['item'].nunique()
_mean_user_item = df_num_items_per_user.mean()
_max_user_item = df_num_items_per_user.max()
_min_user_item = df_num_items_per_user.min()

## 확인용: 아이템별 사용자 수(max, min, mean) -> 5-core
df_num_user_per_item = df_final_input.groupby('item')['user'].nunique()
_mean_item_user = df_num_user_per_item.mean()
_max_item_user = df_num_user_per_item.max()
_min_item_user = df_num_user_per_item.min()

print(f'사용자별 아이템 수:, {round(_mean_user_item, 2)} (평균), {_max_user_item} (최대값), {_min_user_item} (최솟값)')
print(f'아이템별 사용자 수:, {round(_mean_item_user, 2)} (평균), {_max_item_user} (최대값), {_min_item_user} (최솟값)')

사용자별 아이템 수:, 13.65 (평균), 689 (최대값), 5 (최솟값)
아이템별 사용자 수:, 62.91 (평균), 2770 (최대값), 5 (최솟값)


In [ ]:
## 확인용: 사용자 수, 아이템 수, 총 평점 수, 사용자별 아이템 수, 아이템 당 사용자 수, 희소(5-core)
rating_cnt = df_final_input['rating'].count()              ## 총 평점 수
user_cnt = df_final_input['user'].nunique()              ## 사용자 수
item_cnt = df_final_input['item'].nunique()             ## 아이템 수
sparsity = 1 - (rating_cnt / (user_cnt*item_cnt))*100     ## sparsity

print(f'사용자 수: {user_cnt}')
print(f'아이템 수: {item_cnt}')
print(f'총 평점 수: {rating_cnt}')
print(f"Sparsity: {sparsity:.4f} ({sparsity * 100:.2f}%)")

사용자 수: 32927
아이템 수: 7144
총 평점 수: 472877
Sparsity: 0.7990 (79.90%)


In [ ]:
# textrank 시퀀스 생성 후 저장
df_final_input.to_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl')
df_final_input = pd.read_pickle(f'{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl')

print(np.max(list(df_final_input['user_embedding_textrank'].values)))
df_final_input.head(3)

49999


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,AFZUK3MTBIBEDQOPAK3OATUOUKLA,1646112814,5.0,"[7, 65, 157, 1532, 24, 8, 40550, 4482, 112, 67...","[-0.0534612, 0.72937334, 0.8488253, -1.9448721...","[203, 2, 1409, 134, 3, 59, 822, 624, 14, 106, ...","[-0.34872577, 1.6158755, 0.32135147, -3.308696..."
1,AFXMJCVLRCGRAEC7P37FZHEB24UQ,1646112814,4.0,"[2554, 18, 51, 58, 43, 35, 58, 227, 764, 7, 70...","[-1.9726179, -0.33944577, 0.44587654, -1.68373...","[203, 2, 1409, 134, 3, 59, 822, 624, 14, 106, ...","[-0.34872577, 1.6158755, 0.32135147, -3.308696..."
2,AGSP6LSQK32SQEJO3YVVNACPWMSQ,1646112814,4.0,"[7345, 942, 9, 27, 80, 1410, 25, 118, 8620, 9,...","[-1.0965899, -0.9736707, 3.3938568, -1.2914593...","[203, 2, 1409, 134, 3, 59, 822, 624, 14, 106, ...","[-0.34872577, 1.6158755, 0.32135147, -3.308696..."


# Split Train/Test
- 연산 시 array이면 object로 연산이 되지 않아 to_list, float32로 타입 변환함(textrank, bart 모두!!)

In [ ]:
df_input = pd.read_pickle(f"{DATA_SAVE_PATH}/{CATEGORY}_raw_embedding.pkl")
df_input = df_input.rename(columns= {'userID':'user', 'itemID':'item'})

print(df_input.shape)
df_input.head(3)

(472877, 7)


,user,item,rating,user_embedding_textrank,user_embedding_bart,item_embedding_textrank,item_embedding_bart
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[304, 42, 8, 3, 29, 370, 231, 2, 1, 536, 30, 9...","[-1.598509, -0.997596, 0.5327545, -1.525021, 1...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
1,ejFxLGqQcWNLdNByJlIhnQ,XQfwVwDr-v0ZS3_CbbE5Xw,4.0,"[46, 898, 383, 71, 44, 14, 49, 8, 32, 19, 32, ...","[-1.4743725, -1.3923125, -0.2920361, -2.608637...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."
2,f7xa0p_1V9lx53iIGN5Sug,XQfwVwDr-v0ZS3_CbbE5Xw,3.0,"[21, 52, 3, 190, 7, 14, 1, 121, 188, 919, 8, 3...","[-0.21176815, -0.2568603, -1.0278563, -2.40477...","[764, 1, 5411, 11, 1, 112, 2, 45, 158, 5, 6009...","[-1.1874152, -0.1713701, -0.9754306, -1.811321..."


In [ ]:
# 데이터 세트 분할
train_df, test_df = train_test_split(df_input, test_size=0.2, random_state=42)

# 학습 데이터세트
train_users, train_items = train_df[['user']].values, train_df[['item']].values
train_y = train_df['rating'].values

# 테스트 데이터세트
test_users, test_items = test_df[['user']].values, test_df[['item']].values
test_y = test_df['rating'].values

In [ ]:
# train/test data dictionary에 저장
## 주의사항: 임베딩 벡터 타입 변환 필수!! (object이면 연산 XX -> 리스트여야 하고 그 안에 float32가 있어야 가능)
train_dict = {
    "train_user": train_df["user"].values,
    "train_item": train_df["item"].values,
    "train_y": np.array(train_df['rating'].tolist(), dtype=np.float32),
    "train_user_bart": np.array(train_df['user_embedding_bart'].tolist(), dtype=np.float32),
    "train_item_bart": np.array(train_df['item_embedding_bart'].tolist(), dtype=np.float32),
    "train_user_textrank": np.array(train_df['user_embedding_textrank'].tolist(), dtype=np.int32),
    "train_item_textrank": np.array(train_df['item_embedding_textrank'].tolist(), dtype=np.int32)
  }

test_dict = {
    "test_user": test_df["user"].values,
    "test_item": test_df["item"].values,
    "test_y": np.array(test_df['rating'].tolist(), dtype=np.float32),
    "test_user_bart": np.array(test_df['user_embedding_bart'].tolist(), dtype=np.float32),
    "test_item_bart": np.array(test_df['item_embedding_bart'].tolist(), dtype=np.float32),
    "test_user_textrank": np.array(test_df['user_embedding_textrank'].tolist(), dtype=np.int32),
    "test_item_textrank": np.array(test_df['item_embedding_textrank'].tolist(), dtype=np.int32)
  }

In [ ]:
# 딕셔너리를 pickle 파일로 저장
with open(train_path, 'wb') as f:
    pickle.dump(train_dict, f)

with open(test_path, 'wb') as f:
    pickle.dump(test_dict, f)

In [ ]:
# pickle 파일 불러오기
with open(train_path, 'rb') as f:
    train_dict = pickle.load(f)

with open(test_path, 'rb') as f:
    test_dict = pickle.load(f)

# Proposed Model
- Concatenate은 추출형/추상형 요약끼리 유저-아이템 임베딩 벡터 결합하는 방식
- MLP는 300차원으로 textrank, bart 임베딩 차원 맞춰주려고 진행함
- 300차원의 textrank, bart 임베딩에 대해 게이트 벡터를 계산하고 가중치를 고려해 결합함

In [3]:
# user/item 임베딩 행렬 불러오기
user_embedding_matrix = np.load(f'{DATA_SAVE_PATH}/{CATEGORY}_user_glove_dic.npy', allow_pickle=True)
item_embedding_matrix = np.load(f'{DATA_SAVE_PATH}/{CATEGORY}_item_glove_dic.npy', allow_pickle=True)

user_embedding_matrix.shape, item_embedding_matrix.shape

((50001, 300), (50001, 300))

In [4]:
# pickle 파일 불러오기
with open(train_path, 'rb') as f:
    train_dict = pickle.load(f)

with open(test_path, 'rb') as f:
    test_dict = pickle.load(f)

In [5]:
# dict파일 부른 다음에 변수에 할당
train_y = train_dict["train_y"]
train_user_bart = train_dict["train_user_bart"]
train_item_bart = train_dict["train_item_bart"]
train_user_textrank = train_dict["train_user_textrank"]
train_item_textrank = train_dict["train_item_textrank"]


test_y = test_dict["test_y"]
test_user_bart = test_dict["test_user_bart"]
test_item_bart = test_dict["test_item_bart"]
test_user_textrank = test_dict["test_user_textrank"]
test_item_textrank = test_dict["test_item_textrank"]

In [6]:
def point_wise_feed_forward_network(d_model, dff):
  return tf.keras.Sequential([
      tf.keras.layers.Dense(dff, activation='linear'),
      tf.keras.layers.Dense(d_model, activation='linear')
  ])

class CoAttentionBlock(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dff, dropout_rate=0.1, epsilon=1e-6, input_dim=None, **kwargs):
        super(CoAttentionBlock, self).__init__(**kwargs)
        self.multi_head_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, value_dim=key_dim)
        self.dropout = Dropout(dropout_rate)
        self.add = Add()
        self.layer_norm = tf.keras.layers.LayerNormalization(epsilon = epsilon)
        self.ffn = point_wise_feed_forward_network(input_dim, dff)
        self.flatten = Flatten()


    def call(self, inputs):
        text_expanded, image_expanded = inputs
        co_attention = self.multi_head_attention(text_expanded, image_expanded, image_expanded)
        co_attention = self.dropout(co_attention)
        co_attention = self.add([co_attention, text_expanded])
        co_attention = self.layer_norm(co_attention)

        co_attention_ffn = self.ffn(co_attention)
        co_attention_ffn = self.dropout(co_attention_ffn)
        co_attention_ffn = self.add([co_attention_ffn, co_attention])
        co_attention_ffn = self.layer_norm(co_attention_ffn)

        # return self.flatten(co_attention_ffn)
        return co_attention_ffn

class SelfAttentionBlock(tf.keras.layers.Layer):
    def __init__(self, num_heads, key_dim, dff, dropout_rate=0.1,  epsilon=1e-6, input_dim=None, **kwargs):
        super(SelfAttentionBlock, self).__init__(**kwargs)
        self.multi_head_attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, value_dim=key_dim)
        self.dropout = Dropout(dropout_rate)
        self.add = Add()
        self.layer_norm = tf.keras.layers.LayerNormalization(epsilon = epsilon)
        self.ffn = point_wise_feed_forward_network(input_dim, dff)
        self.flatten = Flatten()


    def call(self, inputs):
        information_expanded = inputs
        self_attention = self.multi_head_attention(information_expanded, information_expanded, information_expanded)
        self_attention = self.dropout(self_attention)
        self_attention = self.add([self_attention, information_expanded])
        self_attention = self.layer_norm(self_attention)

        self_attention_ffn = self.ffn(self_attention)
        self_attention_ffn = self.dropout(self_attention_ffn)
        self_attention_ffn = self.add([self_attention_ffn, self_attention])
        self_attention_ffn = self.layer_norm(self_attention_ffn)

        # return self.flatten(self_attention_ffn)
        return self_attention_ffn

In [7]:
def proposed_model(input_dim, textrank_bool = False, bart_bool = False,
                   textrank_user_embedding_matrix = None,
                   textrank_item_embedding_matrix = None,
                   output_dim = 300, num_heads = 4, dff = 1024,
                   fusion_version = 'gmu', gmu_activation = 'sigmoid',
                   mlp_depth=2, mlp_hidden_dim=128, dropout_rate = 0.2):
    """
    [input]
      - textrank_bool: textrank 요약을 반영하는지 여부
      - bart_bool: bart 요약을 반영하는지 여부
      - input_dim: 임베딩 층의 input_dim, 요약의 max_seq_length를 의미함          (※ 각 데이터셋의 평균 패딩 시퀀스 길이임)
      - output_dim: 각 textrank/bart 요약 임베딩 차원                          (=추출/추상형 요약 간 결합 전에 맞춰주어야 하는 차원의 크기)
      - fusion_version: 추출/추상형 요약 간의 결합 방식                           (ex. gmu, concat, inner-product)
      - gmu_activation: gmu로 결합할 때 활성화함수
      - mlp_hidden_dim: mlp 은닉층 차원
      - mlp_depth: mlp 은닉층 깊이
      - dropout_rate: dropout 비율

    [output]
      - model: 모델 객체
    """
    # 0. attn block
    key_dim = output_dim // num_heads

    self_attn_block = SelfAttentionBlock(
        num_heads=num_heads, key_dim=key_dim,
        dropout_rate=dropout_rate,
        dff=dff,
        input_dim=output_dim              # 여기서 input_dim으로 output_dim을 넘겨줘야 함
    )
    
    co_attn_block = CoAttentionBlock(
        num_heads=num_heads, key_dim=key_dim,
        dropout_rate=dropout_rate,
        dff=dff, input_dim=output_dim
    )

    if textrank_bool == True:
        user_textrank_input = Input(shape=(input_dim,), dtype='int32', name='user_textrank_input')
        item_textrank_input = Input(shape=(input_dim,), dtype='int32', name='item_textrank_input')

        # 1. user/item embedding & concat
        user_textrank_embedding = Embedding(
            input_dim = len(textrank_user_embedding_matrix),
            output_dim = 300, input_length = input_dim,
            weights = [textrank_user_embedding_matrix])(user_textrank_input)
        item_textrank_embedding = Embedding(
            input_dim = len(textrank_item_embedding_matrix),
            output_dim = 300, input_length = input_dim,
            weights = [textrank_item_embedding_matrix])(item_textrank_input)

        user_textrank_embedding = GlobalMaxPooling1D()(user_textrank_embedding)
        item_textrank_embedding = GlobalMaxPooling1D()(item_textrank_embedding)

        textrank_concat = Concatenate()([user_textrank_embedding, item_textrank_embedding])
        textrank_proj = Dense(output_dim, activation='linear')(textrank_concat)

        # 2. self multi-head attn
        textrank_input = self_attn_block(tf.expand_dims(textrank_proj, axis=1))      # (batch, 1, output_dim)
        textrank_input = tf.squeeze(textrank_input, axis=1)
    else:
        textrank_input = None

    if bart_bool == True:
        # 1. user/item embedding & concat
        user_bart_input = Input(shape=(768,), name='user_bart_input')
        item_bart_input = Input(shape=(768,), name='item_bart_input')

        bart_concat = Concatenate()([user_bart_input, item_bart_input])
        bart_proj = Dense(output_dim, activation='linear')(bart_concat)

        # 2. self multi-head attn
        bart_proj_expanded = tf.expand_dims(bart_proj, axis=1)                            # (batch, 1, output_dim)
        bart_input = self_attn_block(bart_proj_expanded)
        bart_input = tf.squeeze(bart_input, axis=1)
    else:
        bart_input = None


    # 3. Fusion
    if textrank_bool and bart_bool:
        ## 1) gmu: concat > gate > gated sum
        if fusion_version.lower() == 'gmu':
            concat_inputs = Concatenate(name='concat_inputs')([textrank_input, bart_input])
            gate_sum = Dense(output_dim, activation=gmu_activation, name='gmu_gate')(concat_inputs)

            tanh_textrank = Activation('tanh')(textrank_input)
            tanh_bart = Activation('tanh')(bart_input)

            gmu_output = Add()([
                  Multiply()([gate_sum, tanh_textrank]),
                  Multiply()([Lambda(lambda x: 1 - x)(gate_sum), tanh_bart])
              ])
            combined = gmu_output

        ## 2) concat
        elif fusion_version.lower() == 'concat':
            combined = Concatenate()([textrank_input, bart_input])

        ## 3) inner_product
        elif fusion_version.lower().startswith('inner'):
            combined = Dot(axes=-1)([textrank_input, bart_input])

        ## 4) element-wise
        elif fusion_version.lower().startswith('element'):
            combined = Multiply()([textrank_input, bart_input])
        
        ## 5) attn
        elif fusion_version.lower().startswith('at'):
            # co-attn 위해 다시 expand
            textrank_input_exp = tf.expand_dims(textrank_input, axis=1)  # shape: (None, 1, 300)
            bart_input_exp = tf.expand_dims(bart_input, axis=1)

            co_attn_tr = co_attn_block((textrank_input_exp, bart_input_exp))
            co_attn_br = co_attn_block((bart_input_exp, textrank_input_exp))
            combined = Concatenate(name = 'co_attn_output')([co_attn_tr, co_attn_br])
            combined = tf.squeeze(combined, axis=1)
            
        else:
            raise ValueError("Invalid version")

    elif textrank_bool:
        combined = textrank_input
    elif bart_bool:
        combined = bart_input
    else:
        raise ValueError("At least one of textrank_bool or bart_bool must be True.")


    # 4. MLP -> 평점 예측
    ## (combined의 차원이 'cag' 퓨전 시 2*output_dim이 될 수 있으므로, MLP가 이를 처리할 수 있도록 설계되었는지 확인해야 합니다.)
    mlp_hidden_units = [mlp_hidden_dim // (2 ** i) for i in range(mlp_depth)]
    x = combined
    for i, units in enumerate(mlp_hidden_units):
        x = Dense(units, activation='linear', name=f'mlp_dense_{i+1}')(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate, name=f'dropout_{i+1}')(x)
    rating_pred = Dense(1, activation='relu', name='rating_output')(x)


    # 5. 모델 구성
    input_list = []
    if textrank_bool == True:
        input_list += [user_textrank_input, item_textrank_input]
    if bart_bool == True:
        input_list += [user_bart_input, item_bart_input]

    model = Model(inputs=input_list, outputs=rating_pred, name=f'{fusion_version}_rating_model')

    return model

## Training

### (1)modeling code
- 1번 학습하는 코드 -> 모델 설계가 이상하지 않은지, 각 단계별로 제대로 돌아가는지 확인하기 위함

In [10]:
model = proposed_model(
    textrank_bool=True, bart_bool=True,
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    fusion_version= 'at',
    input_dim=default_params['input_dim'], output_dim=default_params['embedding_dim'],
    dropout_rate=default_params['dropout_rate'],
    mlp_hidden_dim=default_params['mlp_hidden_dim'], mlp_depth=default_params['mlp_depth']
)

model.compile(
    optimizer=Adam(learning_rate=default_params['learning_rate']),
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(),
        tf.keras.metrics.MeanSquaredError()]
)

model.summary()

Model: "at_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 118)]       0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 118)]       0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 118, 300)     15000300    ['user_textrank_input[0][0]']    
                                                                                    

In [14]:
# EarlyStopping
early_stopping = EarlyStopping(
      monitor='val_loss', mode='min', patience=5,
      restore_best_weights=True, verbose=1
    )

train_inputs = [
    train_user_textrank, train_item_textrank, 
    train_user_bart, train_item_bart
]

history = model.fit(
    train_inputs,
    train_dict['train_y'],
    validation_split=0.125,
    epochs=100,
    batch_size=default_params['batch_size'],
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
3357/3357 [==============================] - 780s 231ms/step - loss: 1.1056 - mean_absolute_error: 0.7965 - mean_squared_error: 1.1056 - val_loss: 0.6850 - val_mean_absolute_error: 0.5424 - val_mean_squared_error: 0.6850
Epoch 2/100
3357/3357 [==============================] - 783s 233ms/step - loss: 0.8546 - mean_absolute_error: 0.6937 - mean_squared_error: 0.8546 - val_loss: 0.6470 - val_mean_absolute_error: 0.5367 - val_mean_squared_error: 0.6470
Epoch 3/100
3357/3357 [==============================] - 782s 233ms/step - loss: 0.7845 - mean_absolute_error: 0.6600 - mean_squared_error: 0.7845 - val_loss: 0.7737 - val_mean_absolute_error: 0.5401 - val_mean_squared_error: 0.7737
Epoch 4/100
3357/3357 [==============================] - 782s 233ms/step - loss: 0.7372 - mean_absolute_error: 0.6365 - mean_squared_error: 0.7372 - val_loss: 0.6681 - val_mean_absolute_error: 0.5084 - val_mean_squared_error: 0.6681
Epoch 5/100
3357/3357 [==============================] - 779s 232ms/

In [15]:
test_inputs = [
    test_user_textrank, test_item_textrank, 
    test_user_bart, test_item_bart
]

predicted_ratings = model.predict(test_inputs)
test_metrics = {
    'MAE': mean_absolute_error(test_dict['test_y'], predicted_ratings),
    'MSE': mean_squared_error(test_dict['test_y'], predicted_ratings),
    'RMSE': np.sqrt(mean_squared_error(test_dict['test_y'], predicted_ratings)),
    'MAPE': 100 * mean_absolute_percentage_error(test_dict['test_y'], predicted_ratings)
}

print(test_metrics)

3836/3836 [==============================] - 27s 7ms/step
{'MAE': 0.4955677390098572, 'MSE': 0.572766125202179, 'RMSE': 0.756813137572399, 'MAPE': 16.536104679107666}


### (2)modeling function code
- n번 모델링할 수 있게 함수화한 코드

In [10]:
# 모델 5번 학습 후 성능 저장
def train_proposed_model(
    dict_params,
    textrank_bool=True, bart_bool=True,
    textrank_user_embedding_matrix = None,
    textrank_item_embedding_matrix = None,
    fusion_version = 'gmu', nums_train = 5, verbose = True
):
    """
    [parameter]
      - dict_params: 튜닝된 하이퍼파라미터 -> 딕셔너리 형태
      - textrank_bool: textrank 요약을 반영하는지 여부
      - bart_bool: bart 요약을 반영하는지 여부
      - textrank_user_embedding_matrix: textrank 요약을 반영하는지 여부에 따라 user의 textrank 임베딩 행렬 입력
      - textrank_item_embedding_matrix: textrank 요약을 반영하는지 여부에 따라 item의 textrank 임베딩 행렬 입력
      - fusion_version: 추출형/추상형 요약 융합 방식
      - nums_train: 학습 횟수
      - verbose: model summary 출력 여부

    [output]
      - df_res: 5번의 학습 평가지표를 담은 dataframe
    """
    df_res = pd.DataFrame()

    # EarlyStopping
    early_stopping = EarlyStopping(
        monitor='val_loss', mode='min', patience=5,
        restore_best_weights=True, verbose=1
    )

    # train/test input data
    train_inputs = []
    test_inputs = []

    if textrank_bool:
        train_inputs += [train_user_textrank, train_item_textrank]
        test_inputs += [test_user_textrank, test_item_textrank]
    if bart_bool:
        train_inputs += [train_user_bart, train_item_bart]
        test_inputs += [test_user_bart, test_item_bart]

    # start training
    for i in range(1, nums_train+1):
        tf.keras.backend.clear_session()
        gc.collect()

        try:
            model = proposed_model(
                textrank_bool=textrank_bool, bart_bool=bart_bool,
                textrank_user_embedding_matrix=textrank_user_embedding_matrix,
                textrank_item_embedding_matrix=textrank_item_embedding_matrix,
                fusion_version= fusion_version,
                input_dim=dict_params['input_dim'], output_dim=dict_params['embedding_dim'],
                dropout_rate=dict_params['dropout_rate'],
                mlp_hidden_dim=dict_params['mlp_hidden_dim'], mlp_depth=dict_params['mlp_depth']
            )

            model.compile(
                optimizer=Adam(learning_rate=dict_params['learning_rate']),
                loss=tf.keras.losses.MeanSquaredError(),
                metrics=[
                    tf.keras.metrics.MeanAbsoluteError(),
                    tf.keras.metrics.MeanSquaredError()]
            )

            if verbose:
                print(model.summary())

            print(f"\n✅ 모델 훈련 {i}/{nums_train} 시작...")

            history = model.fit(
                train_inputs, train_dict['train_y'],
                validation_split=0.125, epochs=100,
                batch_size=dict_params['batch_size'],
                callbacks=[early_stopping], verbose=1
            )

        except Exception as e:
            print(f"🚨 훈련 중 오류 발생: {e}")
            print("🔁 CPU로 전환하여 재시도...")

            tf.keras.backend.clear_session()
            gc.collect()

            with tf.device('/CPU:0'):
                model = proposed_model(
                    textrank_bool=textrank_bool, bart_bool=bart_bool,
                    textrank_user_embedding_matrix=textrank_user_embedding_matrix,
                    textrank_item_embedding_matrix=textrank_item_embedding_matrix,
                    input_dim=dict_params['input_dim'], output_dim=dict_params['embedding_dim'],
                    dropout_rate=dict_params['dropout_rate'],
                    mlp_hidden_dim=dict_params['mlp_hidden_dim'], mlp_depth=dict_params['mlp_depth'],
                    fusion_version=dict_params['fusion_version']
                )

                model.compile(
                    optimizer=Adam(learning_rate=dict_params['learning_rate']),
                    loss=tf.keras.losses.MeanSquaredError(),
                    metrics=[
                        tf.keras.metrics.MeanAbsoluteError(),
                        tf.keras.metrics.MeanSquaredError()]
                )

                if verbose:
                    print(model.summary())

                print(f"\n✅ 모델 훈련 {i}/{nums_train} 시작...")

                history = model.fit(
                    train_inputs, train_dict['train_y'],
                    validation_split=0.125, epochs=100,
                    batch_size=dict_params['batch_size'],
                    callbacks=[early_stopping], verbose=1
                )

        # 테스트 평가
        predicted_ratings = model.predict(test_inputs)
        test_metrics = {
            'MAE': mean_absolute_error(test_dict['test_y'], predicted_ratings),
            'MSE': mean_squared_error(test_dict['test_y'], predicted_ratings),
            'RMSE': np.sqrt(mean_squared_error(test_dict['test_y'], predicted_ratings)),
            'MAPE': 100 * mean_absolute_percentage_error(test_dict['test_y'], predicted_ratings)
        }

        print(test_metrics)

        _df_res_temp = pd.DataFrame.from_dict(test_metrics, orient='index', columns=[i])
        df_res = pd.concat([df_res, _df_res_temp], axis=1)

    return df_res

In [12]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=True,
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=3
)
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

# 전체 결과 저장
# RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_proposedmodel_result.csv"
RESULT_FILE_PATH = f"dataset/{CATEGORY}_proposedmodel_result.csv"

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "gmu_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 113)]       0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 113)]       0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 113, 300)     15000300    ['user_textrank_input[0][0]']    
                                                                                   

# Tuning

In [ ]:
# 각 튜닝할 하이퍼파라미터 리스트
tuning_params = {
    'learning_rate': [1e-3, 5e-4, 3e-4, 1e-4, 1e-5],
    'batch_size': [64, 128, 256],
    'dropout_rate': [0, 0.1, 0.3, 0.4, 0.5],
    'mlp_hidden_dim': [64, 128, 256, 512],
    'mlp_depth': [1, 2, 3, 4, 5],
    'embedding_dim': [128, 256, 300, 512]
}

In [19]:
def tuning_params(param_dict, default_params, num_runs=2, save_path='results.csv', verbose=True):
    """
    단일 또는 다중 하이퍼파라미터 조합을 튜닝하는 함수

    [input]
      - param_dict: {'param_name': [values]} (단일 또는 다중 파라미터 모두 지원)
      - default_params: 고정 파라미터
      - num_runs: 각 조합 반복 횟수
      - save_path: 결과 저장 경로
      - verbose: 출력 여부
    [output]
      - df_results: 결과 DataFrame
    """
    keys = list(param_dict.keys())
    values = list(param_dict.values())
    param_combinations = list(itertools.product(*values))
    results = []

    # update dictionary of hyper-parameter
    for combo in param_combinations:
        current_params = default_params.copy()
        for k, v in zip(keys, combo):
            current_params[k] = v

        param_str = ', '.join(f"{k}={v}" for k, v in zip(keys, combo))
        if verbose:
            print(f"\n🔧 튜닝 중: {param_str}\n")
            print("📌 사용된 파라미터:")
            for k, v in sorted(current_params.items()):
                print(f"  - {k}: {v}")

        for i in trange(num_runs, desc=f"Tuning({param_str})"):
            tf.keras.backend.clear_session()
            gc.collect()
            start_time = time.time()

            if verbose:
                print(f"\n🚀 반복 {i+1}/{num_runs} 시작...\n")

            model = proposed_model(
                textrank_bool=True,
                bart_bool=True,
                input_dim=current_params['input_dim'],
                output_dim=current_params['embedding_dim'],
                dropout_rate=current_params['dropout_rate'],
                mlp_hidden_dim=current_params['mlp_hidden_dim'],
                textrank_user_embedding_matrix=user_embedding_matrix,         ## global variable
                textrank_item_embedding_matrix=item_embedding_matrix          ## global variable
            )

            model.compile(
                optimizer=Adam(learning_rate=current_params['learning_rate']),
                loss=tf.keras.losses.MeanSquaredError(),
                metrics=[
                    tf.keras.metrics.MeanAbsoluteError(),
                    tf.keras.metrics.MeanSquaredError()
                ]
            )

            early_stopping = EarlyStopping(
                monitor='val_loss', mode='min', patience=5,
                restore_best_weights=True, verbose=0
            )

            history = model.fit(
                [train_user_textrank, train_item_textrank, train_user_bart, train_item_bart],     ## global variable
                train_dict['train_y'],                                                            ## global variable
                validation_split=0.125,
                epochs=100,
                batch_size=current_params['batch_size'],
                callbacks=[early_stopping],
                verbose=1
            )

            preds = model.predict([test_user_textrank, test_item_textrank, test_user_bart, test_item_bart]) ## global variable

            test_metrics = {
                'Run': i + 1,
                'Train_Time_sec': round(time.time() - start_time, 2),
                'MAE': mean_absolute_error(test_dict['test_y'], preds),
                'MSE': mean_squared_error(test_dict['test_y'], preds),
                'RMSE': np.sqrt(mean_squared_error(test_dict['test_y'], preds)),
                'MAPE': 100 * mean_absolute_percentage_error(test_dict['test_y'], preds),
            }
            test_metrics.update({k: v for k, v in zip(keys, combo)})
            results.append(test_metrics)

            if verbose:
                print(f"\n✅ Run {i+1} | {param_str} | "
                      f"MAE: {test_metrics['MAE']:.4f}, MSE: {test_metrics['MSE']:.4f}, "
                      f"RMSE: {test_metrics['RMSE']:.4f}, MAPE: {test_metrics['MAPE']:.4f}")

    # 결과 저장
    if os.path.exists(save_path):
        df_results = pd.read_csv(save_path)
        print("📁 기존 결과에 추가 저장")
    else:
        df_results = pd.DataFrame()
        print("🆕 새 결과 파일 생성")

    df_results = pd.concat([df_results, pd.DataFrame(results)], axis=0)
    df_results.to_csv(save_path, index=False)
    print(f"\n📌 저장 완료: '{save_path}'")

    return df_results

#### 튜닝 결과

* batch size = 128
* dropout = 0.3
* learning rate = 0.0003
* embedding dim = 300
* mlp dim = 512
* mlp layer = 2

In [ ]:
# 개별 튜닝 후 조합해서 성능 확인
TUNING_SAVE_PATH = f"dataset"
# TUNING_SAVE_PATH = "/content/drive/MyDrive/박민경_논문1/file_metrics"
df_tuned_res = tuning_params(
    param_dict={
            'mlp_hidden_dim': [512],
        },
    default_params=default_params,
    num_runs=2,
    save_path=f"{TUNING_SAVE_PATH}/{CATEGORY}_tuning_res.csv"
)


🔧 튜닝 중: mlp_hidden_dim=512



Tuning(mlp_hidden_dim=512):   0%|          | 0/2 [00:00<?, ?it/s]


🚀 반복 1/2 시작...

Epoch 1/100
3357/3357 [==============================] - 768s 228ms/step - loss: 0.7748 - mean_absolute_error: 0.6434 - mean_squared_error: 0.7748 - val_loss: 0.8404 - val_mean_absolute_error: 0.6262 - val_mean_squared_error: 0.8404
Epoch 2/100
3357/3357 [==============================] - 769s 229ms/step - loss: 0.6878 - mean_absolute_error: 0.6011 - mean_squared_error: 0.6878 - val_loss: 0.7221 - val_mean_absolute_error: 0.5515 - val_mean_squared_error: 0.7221
Epoch 3/100
3357/3357 [==============================] - 772s 230ms/step - loss: 0.6472 - mean_absolute_error: 0.5774 - mean_squared_error: 0.6472 - val_loss: 0.6077 - val_mean_absolute_error: 0.5223 - val_mean_squared_error: 0.6077
Epoch 4/100
3357/3357 [==============================] - 770s 229ms/step - loss: 0.6200 - mean_absolute_error: 0.5609 - mean_squared_error: 0.6200 - val_loss: 0.6463 - val_mean_absolute_error: 0.4910 - val_mean_squared_error: 0.6463
Epoch 5/100
3357/3357 [============================

Tuning(mlp_hidden_dim=512):  50%|█████     | 1/2 [2:59:44<2:59:44, 10784.41s/it]


✅ Run 1 | mlp_hidden_dim=512 | MAE: 0.4868, MSE: 0.5679, RMSE: 0.7536, MAPE: 16.6723

🚀 반복 2/2 시작...

Epoch 1/100
3357/3357 [==============================] - 773s 230ms/step - loss: 0.7743 - mean_absolute_error: 0.6422 - mean_squared_error: 0.7743 - val_loss: 0.7968 - val_mean_absolute_error: 0.5854 - val_mean_squared_error: 0.7968
Epoch 2/100
3357/3357 [==============================] - 773s 230ms/step - loss: 0.6884 - mean_absolute_error: 0.6004 - mean_squared_error: 0.6884 - val_loss: 0.7281 - val_mean_absolute_error: 0.5661 - val_mean_squared_error: 0.7281
Epoch 3/100
3357/3357 [==============================] - 770s 229ms/step - loss: 0.6489 - mean_absolute_error: 0.5773 - mean_squared_error: 0.6489 - val_loss: 0.7570 - val_mean_absolute_error: 0.5641 - val_mean_squared_error: 0.7570
Epoch 4/100
3357/3357 [==============================] - 769s 229ms/step - loss: 0.6202 - mean_absolute_error: 0.5610 - mean_squared_error: 0.6202 - val_loss: 0.7813 - val_mean_absolute_error: 0.587

Tuning(mlp_hidden_dim=512): 100%|██████████| 2/2 [6:23:36<00:00, 11508.45s/it]  


✅ Run 2 | mlp_hidden_dim=512 | MAE: 0.4653, MSE: 0.5852, RMSE: 0.7650, MAPE: 16.7726
📁 기존 결과에 추가 저장

📌 저장 완료: 'dataset/book_tuning_res.csv'


## (1)batch_size

In [ ]:
df_batchsize_res = tuning_params(
    param_dict={'batch_size': [64]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_batchsize_tuning_res.csv"
)

In [ ]:
df_batchsize_res = tuning_params(
    param_dict={'batch_size': [128]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_batchsize_tuning_res.csv"
)

In [ ]:
df_batchsize_res = tuning_params(
    param_dict={'batch_size': [256]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_batchsize_tuning_res.csv"
)

## (2)learning rate

In [ ]:
df_lr_tuned_res = tuning_params(
    param_dict={'learning_rate': [tuning_params['learning_rate'][0]]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_lr_tuning_res.csv"
)

In [ ]:
df_lr_tuned_res = tuning_params(
    param_dict={'learning_rate': [0.0005]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_lr_tuning_res.csv"
)


🔧 튜닝 중: learning_rate = 0.0005



learning_rate=0.0005:   0%|          | 0/2 [00:00<?, ?it/s]


🚀 반복 1/2 시작...

Epoch 1/100
3357/3357 [==============================] - 762s 226ms/step - loss: 0.8887 - mean_absolute_error: 0.7011 - mean_squared_error: 0.8887 - val_loss: 0.9077 - val_mean_absolute_error: 0.6485 - val_mean_squared_error: 0.9077
Epoch 2/100
3357/3357 [==============================] - 764s 227ms/step - loss: 0.7144 - mean_absolute_error: 0.6225 - mean_squared_error: 0.7144 - val_loss: 0.6950 - val_mean_absolute_error: 0.5363 - val_mean_squared_error: 0.6950
Epoch 3/100
3357/3357 [==============================] - 764s 228ms/step - loss: 0.6250 - mean_absolute_error: 0.5737 - mean_squared_error: 0.6250 - val_loss: 0.5747 - val_mean_absolute_error: 0.5125 - val_mean_squared_error: 0.5747
Epoch 4/100
3357/3357 [==============================] - 750s 223ms/step - loss: 0.5702 - mean_absolute_error: 0.5410 - mean_squared_error: 0.5702 - val_loss: 0.5599 - val_mean_absolute_error: 0.4763 - val_mean_squared_error: 0.5599
Epoch 5/100
3357/3357 [============================

learning_rate=0.0005:  50%|█████     | 1/2 [1:54:18<1:54:18, 6858.31s/it]

✅ Run 1 | learning_rate=0.0005 | MAE: 0.4769, MSE: 0.5661, RMSE: 0.7524, MAPE: 16.3735

🚀 반복 2/2 시작...

Epoch 1/100
3357/3357 [==============================] - 767s 228ms/step - loss: 0.8594 - mean_absolute_error: 0.6909 - mean_squared_error: 0.8594 - val_loss: 0.6085 - val_mean_absolute_error: 0.5281 - val_mean_squared_error: 0.6085
Epoch 2/100
3357/3357 [==============================] - 765s 228ms/step - loss: 0.6874 - mean_absolute_error: 0.6086 - mean_squared_error: 0.6874 - val_loss: 0.7891 - val_mean_absolute_error: 0.6201 - val_mean_squared_error: 0.7891
Epoch 3/100
3357/3357 [==============================] - 766s 228ms/step - loss: 0.6099 - mean_absolute_error: 0.5652 - mean_squared_error: 0.6099 - val_loss: 0.5620 - val_mean_absolute_error: 0.4847 - val_mean_squared_error: 0.5620
Epoch 4/100
3357/3357 [==============================] - 763s 227ms/step - loss: 0.5618 - mean_absolute_error: 0.5373 - mean_squared_error: 0.5618 - val_loss: 0.5951 - val_mean_absolute_error: 0.48

learning_rate=0.0005: 100%|██████████| 2/2 [4:40:47<00:00, 8423.94s/it]  

✅ Run 2 | learning_rate=0.0005 | MAE: 0.4856, MSE: 0.5619, RMSE: 0.7496, MAPE: 16.6407
🆕 새 결과 파일 생성


OSError: Cannot save file into a non-existent directory: '\content\drive\MyDrive\박민경_논문1\file_metrics'

In [ ]:
df_lr_tuned_res = tuning_params(
    param_dict={'learning_rate': [0.0003]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_lr_tuning_res.csv"
)


🔧 튜닝 중: learning_rate = 0.0003



learning_rate=0.0003:   0%|          | 0/2 [00:00<?, ?it/s]


🚀 반복 1/2 시작...

Epoch 1/100
3357/3357 [==============================] - 769s 228ms/step - loss: 0.8760 - mean_absolute_error: 0.6995 - mean_squared_error: 0.8760 - val_loss: 1.1016 - val_mean_absolute_error: 0.7480 - val_mean_squared_error: 1.1016
Epoch 2/100
3357/3357 [==============================] - 757s 226ms/step - loss: 0.7264 - mean_absolute_error: 0.6292 - mean_squared_error: 0.7264 - val_loss: 0.6176 - val_mean_absolute_error: 0.5184 - val_mean_squared_error: 0.6176
Epoch 3/100
3357/3357 [==============================] - 751s 224ms/step - loss: 0.6530 - mean_absolute_error: 0.5899 - mean_squared_error: 0.6530 - val_loss: 0.6417 - val_mean_absolute_error: 0.5084 - val_mean_squared_error: 0.6417
Epoch 4/100
3357/3357 [==============================] - 751s 224ms/step - loss: 0.6045 - mean_absolute_error: 0.5631 - mean_squared_error: 0.6045 - val_loss: 0.6103 - val_mean_absolute_error: 0.4738 - val_mean_squared_error: 0.6103
Epoch 5/100
3357/3357 [============================

learning_rate=0.0003:  50%|█████     | 1/2 [2:31:05<2:31:05, 9065.73s/it]

✅ Run 1 | learning_rate=0.0003 | MAE: 0.4772, MSE: 0.5626, RMSE: 0.7501, MAPE: 16.3527

🚀 반복 2/2 시작...

Epoch 1/100
3357/3357 [==============================] - 750s 223ms/step - loss: 0.8668 - mean_absolute_error: 0.6949 - mean_squared_error: 0.8668 - val_loss: 0.7167 - val_mean_absolute_error: 0.5541 - val_mean_squared_error: 0.7167
Epoch 2/100
3357/3357 [==============================] - 753s 224ms/step - loss: 0.7340 - mean_absolute_error: 0.6328 - mean_squared_error: 0.7340 - val_loss: 0.7556 - val_mean_absolute_error: 0.5977 - val_mean_squared_error: 0.7556
Epoch 3/100
3357/3357 [==============================] - 753s 224ms/step - loss: 0.6601 - mean_absolute_error: 0.5939 - mean_squared_error: 0.6601 - val_loss: 0.6121 - val_mean_absolute_error: 0.5044 - val_mean_squared_error: 0.6121
Epoch 4/100
3357/3357 [==============================] - 754s 225ms/step - loss: 0.6076 - mean_absolute_error: 0.5649 - mean_squared_error: 0.6076 - val_loss: 0.5626 - val_mean_absolute_error: 0.51

learning_rate=0.0003: 100%|██████████| 2/2 [4:38:23<00:00, 8351.63s/it]  

✅ Run 2 | learning_rate=0.0003 | MAE: 0.5021, MSE: 0.5560, RMSE: 0.7457, MAPE: 16.9948
📁 기존 결과에 추가 저장

📌 저장 완료: 'dataset/book_lr_tuning_res.csv'


In [ ]:
df_lr_tuned_res.head(10)

,Run,MAE,MSE,RMSE,MAPE,batch_size,Train_Time_sec,learning_rate
0,1,0.499414,0.572709,0.756775,16.652404,0.00100,6229.47,NaN
1,1,0.533986,0.623426,0.789573,18.477435,0.00001,11646.14,NaN
2,1,0.485314,0.580899,0.762167,16.372746,0.00050,7753.68,NaN
3,1,0.474607,0.571544,0.756005,16.205609,0.00030,9418.78,NaN
0,1,0.477234,0.562617,0.750078,16.352673,NaN,9065.59,0.0003
1,2,0.502135,0.556033,0.745676,16.994818,NaN,7637.24,0.0003


## (3)dropout

In [ ]:
df_dr_tuned_res = tuning_params(
    param_dict={'dropout_rate': [0.1]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_dr_tuning_res.csv"
)


🔧 튜닝 중: dropout_rate = 0.1



dropout_rate=0.1:   0%|          | 0/2 [00:00<?, ?it/s]


🚀 반복 1/2 시작...

Epoch 1/100
3357/3357 [==============================] - 758s 225ms/step - loss: 0.7977 - mean_absolute_error: 0.6559 - mean_squared_error: 0.7977 - val_loss: 0.6348 - val_mean_absolute_error: 0.5775 - val_mean_squared_error: 0.6348
Epoch 2/100
3357/3357 [==============================] - 772s 230ms/step - loss: 0.7045 - mean_absolute_error: 0.6101 - mean_squared_error: 0.7045 - val_loss: 0.6446 - val_mean_absolute_error: 0.5291 - val_mean_squared_error: 0.6446
Epoch 3/100
3357/3357 [==============================] - 772s 230ms/step - loss: 0.6682 - mean_absolute_error: 0.5898 - mean_squared_error: 0.6682 - val_loss: 0.6233 - val_mean_absolute_error: 0.5025 - val_mean_squared_error: 0.6233
Epoch 4/100
3357/3357 [==============================] - 769s 229ms/step - loss: 0.6390 - mean_absolute_error: 0.5731 - mean_squared_error: 0.6390 - val_loss: 0.5808 - val_mean_absolute_error: 0.5224 - val_mean_squared_error: 0.5808
Epoch 5/100
3357/3357 [============================

dropout_rate=0.1:  50%|█████     | 1/2 [3:00:01<3:00:01, 10801.84s/it]

✅ Run 1 | dropout_rate=0.1 | MAE: 0.4881, MSE: 0.5661, RMSE: 0.7524, MAPE: 17.0769

🚀 반복 2/2 시작...

Epoch 1/100
3357/3357 [==============================] - 761s 226ms/step - loss: 0.7946 - mean_absolute_error: 0.6534 - mean_squared_error: 0.7946 - val_loss: 0.6443 - val_mean_absolute_error: 0.5752 - val_mean_squared_error: 0.6443
Epoch 2/100
3357/3357 [==============================] - 759s 226ms/step - loss: 0.7070 - mean_absolute_error: 0.6113 - mean_squared_error: 0.7070 - val_loss: 0.6319 - val_mean_absolute_error: 0.5496 - val_mean_squared_error: 0.6319
Epoch 3/100
3357/3357 [==============================] - 761s 227ms/step - loss: 0.6667 - mean_absolute_error: 0.5894 - mean_squared_error: 0.6667 - val_loss: 0.6066 - val_mean_absolute_error: 0.5068 - val_mean_squared_error: 0.6066
Epoch 4/100
3357/3357 [==============================] - 761s 227ms/step - loss: 0.6379 - mean_absolute_error: 0.5734 - mean_squared_error: 0.6379 - val_loss: 0.5934 - val_mean_absolute_error: 0.4950 -

dropout_rate=0.1: 100%|██████████| 2/2 [5:07:12<00:00, 9216.31s/it]   

✅ Run 2 | dropout_rate=0.1 | MAE: 0.5053, MSE: 0.5781, RMSE: 0.7603, MAPE: 17.0273
📁 기존 결과에 추가 저장

📌 저장 완료: 'dataset/book_dr_tuning_res.csv'


In [ ]:
df_dr_tuned_res = tuning_params(
    param_dict={'dropout_rate': [0.3]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_dr_tuning_res.csv"
)


🔧 튜닝 중: dropout_rate = 0.3



dropout_rate=0.3:   0%|          | 0/2 [00:00<?, ?it/s]


🚀 반복 1/2 시작...

Epoch 1/100
3357/3357 [==============================] - 752s 223ms/step - loss: 1.0500 - mean_absolute_error: 0.7831 - mean_squared_error: 1.0500 - val_loss: 0.8470 - val_mean_absolute_error: 0.6197 - val_mean_squared_error: 0.8470
Epoch 2/100
3357/3357 [==============================] - 752s 224ms/step - loss: 0.8733 - mean_absolute_error: 0.7080 - mean_squared_error: 0.8733 - val_loss: 0.7870 - val_mean_absolute_error: 0.5883 - val_mean_squared_error: 0.7870
Epoch 3/100
3357/3357 [==============================] - 753s 224ms/step - loss: 0.8144 - mean_absolute_error: 0.6807 - mean_squared_error: 0.8144 - val_loss: 0.7151 - val_mean_absolute_error: 0.5580 - val_mean_squared_error: 0.7151
Epoch 4/100
3357/3357 [==============================] - 753s 224ms/step - loss: 0.7700 - mean_absolute_error: 0.6589 - mean_squared_error: 0.7700 - val_loss: 0.6825 - val_mean_absolute_error: 0.5388 - val_mean_squared_error: 0.6825
Epoch 5/100
3357/3357 [============================

dropout_rate=0.3:  50%|█████     | 1/2 [3:34:00<3:34:00, 12840.62s/it]

✅ Run 1 | dropout_rate=0.3 | MAE: 0.4804, MSE: 0.5714, RMSE: 0.7559, MAPE: 16.4942

🚀 반복 2/2 시작...

Epoch 1/100
3357/3357 [==============================] - 762s 226ms/step - loss: 1.0606 - mean_absolute_error: 0.7862 - mean_squared_error: 1.0606 - val_loss: 0.7669 - val_mean_absolute_error: 0.5901 - val_mean_squared_error: 0.7669
Epoch 2/100
3357/3357 [==============================] - 764s 228ms/step - loss: 0.8724 - mean_absolute_error: 0.7082 - mean_squared_error: 0.8724 - val_loss: 0.8181 - val_mean_absolute_error: 0.5967 - val_mean_squared_error: 0.8181
Epoch 3/100
3357/3357 [==============================] - 764s 227ms/step - loss: 0.8163 - mean_absolute_error: 0.6815 - mean_squared_error: 0.8163 - val_loss: 0.7222 - val_mean_absolute_error: 0.5682 - val_mean_squared_error: 0.7222
Epoch 4/100
3357/3357 [==============================] - 765s 228ms/step - loss: 0.7702 - mean_absolute_error: 0.6584 - mean_squared_error: 0.7702 - val_loss: 0.7348 - val_mean_absolute_error: 0.5745 -

dropout_rate=0.3: 100%|██████████| 2/2 [7:35:53<00:00, 13676.87s/it]  

✅ Run 2 | dropout_rate=0.3 | MAE: 0.4718, MSE: 0.5805, RMSE: 0.7619, MAPE: 16.3430
📁 기존 결과에 추가 저장

📌 저장 완료: 'dataset/book_dr_tuning_res.csv'


In [ ]:
df_dr_tuned_res = tuning_params(
    param_dict={'dropout_rate': [0.4]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_dr_tuning_res.csv"
)

In [ ]:
df_dr_tuned_res = tuning_params(
    param_dict={'dropout_rate': [0.5]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_dr_tuning_res.csv"
)

## (4)mlp hidden dim

In [ ]:
df_mlpdim_tuned_res = tuning_params(
    param_dict={'mlp_hidden_dim': [64]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdim_tuning_res.csv"
)

In [ ]:
df_mlpdim_tuned_res = tuning_params(
    param_dict={'mlp_hidden_dim': [256]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdim_tuning_res.csv"
)

In [ ]:
df_mlpdim_tuned_res = tuning_params(
    param_dict={'mlp_hidden_dim': [512]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdim_tuning_res.csv"
)


🔧 튜닝 중: mlp_hidden_dim = 512



mlp_hidden_dim=512:   0%|          | 0/1 [00:00<?, ?it/s]


🚀 반복 1/1 시작...

Epoch 1/100
3357/3357 [==============================] - 771s 229ms/step - loss: 0.7808 - mean_absolute_error: 0.6460 - mean_squared_error: 0.7808 - val_loss: 0.7126 - val_mean_absolute_error: 0.5303 - val_mean_squared_error: 0.7126
Epoch 2/100
3357/3357 [==============================] - 764s 227ms/step - loss: 0.6901 - mean_absolute_error: 0.6019 - mean_squared_error: 0.6901 - val_loss: 0.7500 - val_mean_absolute_error: 0.5580 - val_mean_squared_error: 0.7500
Epoch 3/100
3357/3357 [==============================] - 767s 229ms/step - loss: 0.6509 - mean_absolute_error: 0.5797 - mean_squared_error: 0.6509 - val_loss: 0.6348 - val_mean_absolute_error: 0.5202 - val_mean_squared_error: 0.6348
Epoch 4/100
3357/3357 [==============================] - 769s 229ms/step - loss: 0.6214 - mean_absolute_error: 0.5623 - mean_squared_error: 0.6214 - val_loss: 0.6680 - val_mean_absolute_error: 0.5301 - val_mean_squared_error: 0.6680
Epoch 5/100
3357/3357 [============================

## (5)mlp layer

In [ ]:
df_mlpdepth_tuned_res = tuning_params(
    param_dict={'mlp_depth': [1]},
    default_params=default_params,
    num_runs=1,
    save_path=f"dataset/{CATEGORY}_mlpdepth_tuning_res.csv"
)


🔧 튜닝 중: mlp_depth=1



Tuning(mlp_depth=1):   0%|          | 0/1 [00:00<?, ?it/s]


🚀 반복 1/1 시작...

Epoch 1/100
3357/3357 [==============================] - 769s 229ms/step - loss: 0.9023 - mean_absolute_error: 0.7129 - mean_squared_error: 0.9023 - val_loss: 0.6548 - val_mean_absolute_error: 0.5405 - val_mean_squared_error: 0.6548
Epoch 2/100
3357/3357 [==============================] - 765s 228ms/step - loss: 0.7804 - mean_absolute_error: 0.6581 - mean_squared_error: 0.7804 - val_loss: 0.9694 - val_mean_absolute_error: 0.6718 - val_mean_squared_error: 0.9694
Epoch 3/100
3357/3357 [==============================] - 766s 228ms/step - loss: 0.7356 - mean_absolute_error: 0.6356 - mean_squared_error: 0.7356 - val_loss: 0.5970 - val_mean_absolute_error: 0.5245 - val_mean_squared_error: 0.5970
Epoch 4/100
3357/3357 [==============================] - 767s 229ms/step - loss: 0.6993 - mean_absolute_error: 0.6154 - mean_squared_error: 0.6993 - val_loss: 0.6961 - val_mean_absolute_error: 0.5469 - val_mean_squared_error: 0.6961
Epoch 5/100
3357/3357 [============================

Tuning(mlp_depth=1): 100%|██████████| 1/1 [2:58:09<00:00, 10689.17s/it]

✅ Run 1 | mlp_depth=1 | MAE: 0.5127, MSE: 0.5645, RMSE: 0.7513, MAPE: 16.8894
📁 기존 결과에 추가 저장

📌 저장 완료: 'dataset/book_mlpdepth_tuning_res.csv'


In [ ]:
df_mlpdepth_tuned_res = tuning_params(
    param_dict={'mlp_depth': [3]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdepth_tuning_res.csv"
)

In [ ]:
df_mlpdepth_tuned_res = tuning_params(
    param_dict={'mlp_depth': [4]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdepth_tuning_res.csv"
)

In [ ]:
df_mlpdepth_tuned_res = tuning_params(
    param_dict={'mlp_depth': [5]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_mlpdepth_tuning_res.csv"
)

## (6)embedding dim

In [ ]:
df_embedding_tuned_res = tuning_params(
    param_dict={'embedding_dim': [128]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_emb_tuning_res.csv"
)

In [ ]:
df_embedding_tuned_res = tuning_params(
    param_dict={'embedding_dim': [256]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_emb_tuning_res.csv"
)

In [ ]:
df_embedding_tuned_res = tuning_params(
    param_dict={'embedding_dim': [512]},
    default_params=default_params,
    num_runs=2,
    save_path=f"dataset/{CATEGORY}_emb_tuning_res.csv"
)

# Ablation Study

## (1)only TextRank

In [ ]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=False,
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=3
)

# 전체 결과 저장
# RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_proposedmodel_result.csv"
RESULT_FILE_PATH = f"dataset/{CATEGORY}_onlytr_result.csv"

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "gmu_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 44, 300)      15000300    ['user_textrank_input[0][0]']    
                                                                                   

## (2)only BART

In [ ]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=False, bart_bool=True,
    nums_train=3, verbose = False
)

# 전체 결과 저장
RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_onlybr_result.csv"
# RESULT_FILE_PATH = f"dataset/{CATEGORY}_onlybr_result.csv"
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

NameError: name 'train_proposed_model' is not defined

# Fusion experiment

## (1)concat

In [ ]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=True, fusion_version='concat',
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=1
)

# 전체 결과 저장
# RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_concat_result.csv"
RESULT_FILE_PATH = f"dataset/{CATEGORY}_concat_result.csv"
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "concat_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 44, 300)      15000300    ['user_textrank_input[0][0]']    
                                                                                

KeyboardInterrupt: 

## (2)element-wise

In [ ]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=True, fusion_version='element_wise',
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=1
)

# 전체 결과 저장
# RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_element_result.csv"
RESULT_FILE_PATH = f"dataset/{CATEGORY}_element_result.csv"
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "element_wise_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 44, 300)      15000300    ['user_textrank_input[0][0]']    
                                                                          

KeyboardInterrupt: 

## (3)inner-product

In [ ]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=True, fusion_version='inner_product',
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=1
)

# 전체 결과 저장
RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_inner_product.csv"
# RESULT_FILE_PATH = f"dataset/{CATEGORY}_inner_product.csv"
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "inner_product_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 44)]        0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 44, 300)      15000300    ['user_textrank_input[0][0]']    
                                                                         

KeyboardInterrupt: 

In [ ]:
df_res

,1,2
MAE,0.718088,0.703768
MSE,1.172626,1.174711
RMSE,1.082879,1.083841
MAPE,32.254776,33.728409


## (4)Co-attn

In [16]:
df_res = train_proposed_model(
    dict_params=default_params,
    textrank_bool=True, bart_bool=True, fusion_version='attention',
    textrank_user_embedding_matrix=user_embedding_matrix,
    textrank_item_embedding_matrix=item_embedding_matrix,
    nums_train=3
)

# 전체 결과 저장
# RESULT_FILE_PATH = f"/content/drive/MyDrive/박민경_논문1/file_metrics/{CATEGORY}_inner_product.csv"
RESULT_FILE_PATH = f"dataset/{CATEGORY}_attn_res.csv"
df_res.columns = [f"{i+1}" for i in range(df_res.shape[1])]

if os.path.exists(RESULT_FILE_PATH):
    df_output = pd.concat([pd.read_csv(RESULT_FILE_PATH, index_col=0), df_res], axis=1)
    df_output.to_csv(RESULT_FILE_PATH, index=True)
    print("📁 기존 결과에 추가 저장")
else:
    df_res.to_csv(RESULT_FILE_PATH, index=True)
    print("🆕 새 결과 파일 저장")

Model: "attention_rating_model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 user_textrank_input (InputLaye  [(None, 113)]       0           []                               
 r)                                                                                               
                                                                                                  
 item_textrank_input (InputLaye  [(None, 113)]       0           []                               
 r)                                                                                               
                                                                                                  
 embedding (Embedding)          (None, 113, 300)     15000300    ['user_textrank_input[0][0]']    
                                                                             